# E4: Feature Selection Funnel — L0 (Data Quality & Compliance)

**Spec sections**: 2.4, 2.5, 6.0, 6.1, 6.8  
**Input**: master table (DEV window), `sample_status ∈ {GOOD, BAD, INDETERMINATE}`; `config/feature_dict.csv`, role = FEATURE only.

In [0]:
import sys, os, hashlib, shutil, tempfile, json, ast, time
from datetime import datetime

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, os.path.join(PROJECT_ROOT, "src"))

import numpy as np
import pandas as pd
import common

cfg = common.load_config(os.path.join(PROJECT_ROOT, "config", "config.yaml"))
ARTIFACTS = common.artifacts_path(cfg)
REPORTS   = os.path.join(ARTIFACTS, "reports")
os.makedirs(REPORTS, exist_ok=True)

cat = cfg["databricks"]["catalog"]
sch = cfg["databricks"]["schema"]
MASTER_TBL = f"{cat}.{sch}.acard_master"
FLOG_TBL   = f"{cat}.{sch}.acard_feature_log"

L0 = cfg["funnel"]["L0"]
DEV_START, DEV_END = cfg["windows"]["DEV"]
target_col = cfg["columns"]["target"]
status_col = cfg["columns"]["status"]

# Funnel config hash (stored in run summary per AGENTS.md)
funnel_config_hash = hashlib.sha256(
    json.dumps(cfg["funnel"], sort_keys=True).encode()
).hexdigest()

# Watch variables minus feature_override (spec TBC-08: AGE is a feature)
_watch_lower = {v.lower() for v in cfg["column_roles"].get("watch", [])} \
             - {v.lower() for v in cfg["column_roles"].get("feature_override", [])}

_TMP = tempfile.mkdtemp(prefix="e4_")
_t0 = time.time()

print(f"Config hash       : {common.config_hash(cfg)[:12]}")
print(f"Funnel config SHA : {funnel_config_hash[:12]}")
print(f"Master table      : {MASTER_TBL}")
print(f"Feature log table : {FLOG_TBL}")
print(f"DEV window        : [{DEV_START}, {DEV_END})")
print(f"L0 thresholds     : single_value={L0['max_single_value_share']}, "
      f"missing={L0['max_missing_rate']}, info_ratio={L0['missing_informative_ratio']}, "
      f"info_min_bad={L0['missing_informative_min_bad']}")
print(f"Watch vars (excl) : {_watch_lower}")

In [0]:
# ── Load master table ────────────────────────────────────────────────
master = spark.table(MASTER_TBL).toPandas()
print(f"Master rows: {len(master):,}")

dev = master[master["window"] == "DEV"].copy()
print(f"DEV rows   : {len(dev):,}")

# L0 input: DEV GOOD + BAD + INDETERMINATE (spec 6.1)
dev_l0 = dev[dev[status_col].isin(["GOOD", "BAD", "INDETERMINATE"])].copy()
print(f"DEV G+B+I  : {len(dev_l0):,}")

# GOOD + BAD only — for informative-missing check (label-based)
dev_gb = dev[dev[status_col].isin(cfg["labelled_statuses"])].copy()
common.assert_labelled(dev_gb, cfg)
print(f"DEV G+B    : {len(dev_gb):,}  |  Bads: {(dev_gb[target_col]==1).sum()}")

In [0]:
# ── Load and validate feature dictionary ─────────────────────────
fdict = pd.read_csv(os.path.join(PROJECT_ROOT, "config", "feature_dict.csv"))
print(f"Feature dictionary: {len(fdict)} rows")
print(f"Roles:\n{fdict['role'].value_counts().to_string()}\n")

feat_df = fdict[fdict["role"] == "FEATURE"].copy().reset_index(drop=True)
print(f"FEATURE candidates: {len(feat_df)}")

# Which features actually exist in the data?
feat_in_data = [f for f in feat_df["feature"] if f in dev_l0.columns]
feat_not_in_data = [f for f in feat_df["feature"] if f not in dev_l0.columns]
if feat_not_in_data:
    print(f"\u26a0 {len(feat_not_in_data)} FEATURE(s) not in master data: {feat_not_in_data[:5]}...")
print(f"Features in data : {len(feat_in_data)}")

# ── Validation (per user specification) ─────────────────────────
issues = []
for _, r in feat_df.iterrows():
    f = r["feature"]
    ad = str(r.get("available_at_decision", "")).strip()
    if ad not in ("Y", "N"):
        issues.append(f"{f}: available_at_decision='{ad}' — must be Y or N")
    co = str(r.get("compliance_ok", "")).strip()
    if co not in ("Y", "N"):
        issues.append(f"{f}: compliance_ok='{co}' — must be Y or N")
    ed = str(r.get("expected_direction", "")).strip()
    if ed not in ("+", "-", "0", "?", "", "nan"):
        issues.append(f"{f}: expected_direction='{ed}' — must be +, -, 0, ? or empty")
    sc = str(r.get("special_codes", "")).strip()
    if sc and sc != "nan":
        try:
            p = ast.literal_eval(sc)
            if not isinstance(p, (list, tuple)):
                issues.append(f"{f}: special_codes='{sc}' — not a list")
        except Exception:
            issues.append(f"{f}: special_codes='{sc}' — cannot parse")
    dcd = str(r.get("definition_change_date", "")).strip()
    if dcd and dcd != "nan":
        try:
            pd.to_datetime(dcd)
        except Exception:
            issues.append(f"{f}: definition_change_date='{dcd}' — invalid date")

if issues:
    oq_path = os.path.join(REPORTS, "open_questions.md")
    with open(oq_path, "a") as f:
        f.write(f"\n## E4 L0 Validation Issues ({datetime.now().isoformat()})\n\n")
        for i in issues:
            f.write(f"- {i}\n")
    print(f"\n\u274c {len(issues)} issues \u2192 {oq_path}")
    for i in issues[:15]:
        print(f"  \u2022 {i}")
    raise ValueError(f"Feature dict has {len(issues)} issues. Fix before proceeding.")
print("\u2705 All FEATURE rows pass validation")

In [0]:
# ── L0: Data Quality and Compliance Rules (spec 6.1) ─────────
# Rules applied in order; first failure drops the feature.

n_total = len(dev_l0)
fdict_idx = feat_df.set_index("feature")
dev_start_dt = pd.to_datetime(DEV_START)
dev_end_dt   = pd.to_datetime(DEV_END)

def _is_watch_derived(name):
    """True if the feature name contains a watch variable name (excl. age)."""
    nl = name.lower()
    return any(w in nl for w in _watch_lower)

records = []
for feat in feat_df["feature"]:
    info = fdict_idx.loc[feat]
    rec = {"feature": feat, "L0_pass": "Y", "L0_reason": None,
           "single_value_share": np.nan, "missing_rate": np.nan,
           "missing_bad_rate_ratio": np.nan}

    if feat not in dev_l0.columns:
        rec.update(L0_pass="N", L0_reason="NOT_IN_DATA")
        records.append(rec)
        continue

    col = dev_l0[feat]
    non_null = col.dropna()
    rec["single_value_share"] = round(
        (non_null.value_counts().iloc[0] / n_total) if len(non_null) > 0 else 0.0, 6)
    rec["missing_rate"] = round(col.isna().sum() / n_total, 6)

    # ── Rule a: available_at_decision = N → TIMING_LEAKAGE ───────
    if str(info.get("available_at_decision", "")).strip() == "N":
        rec.update(L0_pass="N", L0_reason="TIMING_LEAKAGE")
        records.append(rec)
        continue

    # ── Rule b: compliance_ok = N or watch-derived → COMPLIANCE ──
    if str(info.get("compliance_ok", "")).strip() == "N" or _is_watch_derived(feat):
        rec.update(L0_pass="N", L0_reason="COMPLIANCE")
        records.append(rec)
        continue

    # ── Rule c: single_value_share > threshold → SINGLE_VALUE ───
    if rec["single_value_share"] > L0["max_single_value_share"]:
        rec.update(L0_pass="N", L0_reason="SINGLE_VALUE")
        records.append(rec)
        continue

    # ── Rule d: missing_rate > threshold → MISSING ─────────────
    if rec["missing_rate"] > L0["max_missing_rate"]:
        informative = False
        if feat in dev_gb.columns:
            gb_col = dev_gb[feat]
            mask_m = gb_col.isna()
            n_m_bad = (dev_gb.loc[mask_m, target_col] == 1).sum()
            n_m = mask_m.sum()
            n_p = (~mask_m).sum()
            if n_m > 0 and n_p > 0 and n_m_bad >= L0["missing_informative_min_bad"]:
                br_m = n_m_bad / n_m
                br_p = (dev_gb.loc[~mask_m, target_col] == 1).sum() / n_p
                if br_p > 0:
                    ratio = max(br_m / br_p, br_p / br_m)
                    rec["missing_bad_rate_ratio"] = round(ratio, 4)
                    if ratio >= L0["missing_informative_ratio"]:
                        informative = True
        if informative:
            rec["L0_reason"] = "INFORMATIVE_MISSING"
        else:
            rec.update(L0_pass="N", L0_reason="MISSING")
            records.append(rec)
            continue

    # ── Rule e: definition_change_date inside DEV → DEFINITION_CHANGE ─
    dcd = str(info.get("definition_change_date", "")).strip()
    restated = str(info.get("restated", "")).strip().upper()
    if dcd and dcd != "nan":
        try:
            dt = pd.to_datetime(dcd)
            if dev_start_dt <= dt < dev_end_dt and restated != "Y":
                rec.update(L0_pass="N", L0_reason="DEFINITION_CHANGE")
                records.append(rec)
                continue
        except Exception:
            pass

    records.append(rec)

# ── Build feature log ────────────────────────────────────────────
flog = feat_df[["feature", "source", "dtype"]].merge(
    pd.DataFrame(records), on="feature", how="left")
flog["dropped_at"] = pd.NA
flog["drop_rule"] = pd.NA
flog.loc[flog["L0_pass"] == "N", "dropped_at"] = "L0"
flog.loc[flog["L0_pass"] == "N", "drop_rule"] = flog.loc[flog["L0_pass"] == "N", "L0_reason"]

n_pass = (flog["L0_pass"] == "Y").sum()
n_fail = (flog["L0_pass"] == "N").sum()
print(f"L0 results: {n_pass} pass, {n_fail} dropped\n")
print("Drop reasons:")
print(flog[flog["L0_pass"] == "N"]["L0_reason"].value_counts().to_string())
n_info = (flog["L0_reason"] == "INFORMATIVE_MISSING").sum()
if n_info:
    print(f"\nInformative missing (kept): {n_info}")
print(f"\nSurvivors by source:")
print(flog[flog["L0_pass"] == "Y"]["source"].value_counts().to_string())

In [0]:
# ── Save feature log ─────────────────────────────────────────────
# 1. Delta table
sdf = spark.createDataFrame(flog)
sdf.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(FLOG_TBL)
print(f"Saved table: {FLOG_TBL}  ({spark.table(FLOG_TBL).count()} rows)")

# 2. CSV to artifacts
csv_path = os.path.join(ARTIFACTS, "feature_log.csv")
flog.to_csv(csv_path, index=False)
print(f"Saved CSV : {csv_path}")

# ── Report: reports/03_L0.xlsx ──────────────────────────────────
tmp_xl = os.path.join(_TMP, "03_L0.xlsx")
with pd.ExcelWriter(tmp_xl, engine="openpyxl") as writer:
    # Sheet 1: meta
    meta = pd.DataFrame([{
        "module": "E4_L0", "run_time": datetime.now().isoformat(),
        "config_hash": common.config_hash(cfg),
        "funnel_config_hash": funnel_config_hash,
        "dev_rows_gbi": len(dev_l0), "dev_rows_gb": len(dev_gb),
        "features_in": len(flog), "features_pass": int(n_pass), "features_drop": int(n_fail),
    }])
    meta.to_excel(writer, sheet_name="meta", index=False)

    # Sheet 2: counts by reason
    reason_cts = flog.groupby("L0_reason").size().reset_index(name="count").sort_values("count", ascending=False)
    reason_cts.to_excel(writer, sheet_name="counts_by_reason", index=False)

    # Sheet 3: counts by source
    src_pass = flog.groupby(["source", "L0_pass"]).size().unstack(fill_value=0).reset_index()
    src_pass.to_excel(writer, sheet_name="counts_by_source", index=False)

    # Sheet 4: full per-feature table
    flog.sort_values(["L0_pass", "source", "feature"], ascending=[False, True, True]).to_excel(
        writer, sheet_name="feature_detail", index=False)

shutil.copy2(tmp_xl, os.path.join(REPORTS, "03_L0.xlsx"))
print(f"Saved report: {REPORTS}/03_L0.xlsx")

In [0]:
# ── Acceptance Criteria (E4) ───────────────────────────────────
print("=" * 60)
print("ACCEPTANCE CRITERIA (E4 — L0)")
print("=" * 60)
checks = []

# 1. Every FEATURE has L0_pass
checks.append(("Every FEATURE has L0_pass", flog["L0_pass"].notna().all()))

# 2. Every dropped feature has dropped_at and drop_rule
dropped = flog[flog["L0_pass"] == "N"]
if len(dropped):
    checks.append(("Dropped features have dropped_at + drop_rule",
                    (dropped["dropped_at"].notna() & dropped["drop_rule"].notna()).all()))
else:
    checks.append(("Dropped features have dropped_at + drop_rule (none dropped)", True))

# 3. No WATCH/KEY/FLAG/PERFORMANCE column passes L0
#    (guaranteed by construction: flog only contains role=FEATURE rows)
non_feat = set(fdict[fdict["role"].isin(["WATCH", "KEY", "FLAG", "PERFORMANCE"])]["feature"])
leaked = flog[(flog["feature"].isin(non_feat)) & (flog["L0_pass"] == "Y")]
checks.append(("No WATCH/KEY/FLAG/PERFORMANCE column passes", len(leaked) == 0))

# 4. Only DEV GOOD+BAD+INDETERMINATE rows used
checks.append(("Only DEV G+B+I rows used (by construction)", True))

# 5. feature_log.csv exported
checks.append(("feature_log.csv exported", os.path.exists(os.path.join(ARTIFACTS, "feature_log.csv"))))

# 6. Feature log table row count matches
tbl_ct = spark.table(FLOG_TBL).count()
checks.append((f"Feature log table rows = {tbl_ct}", tbl_ct == len(flog)))

for desc, ok in checks:
    status = "\u2705 PASS" if ok else "\u274c FAIL"
    print(f"  {status} : {desc}")
    assert ok, f"ACCEPTANCE FAILED: {desc}"

# ── Run summary ────────────────────────────────────────────────
summary = {
    "module": "E4_L0",
    "run_time": datetime.now().isoformat(),
    "elapsed_s": round(time.time() - _t0, 1),
    "config_hash": common.config_hash(cfg),
    "funnel_config_hash": funnel_config_hash,
    "dev_rows_gbi": len(dev_l0),
    "dev_rows_gb": len(dev_gb),
    "features_total": len(flog),
    "features_pass": int(n_pass),
    "features_drop": int(n_fail),
    "drop_reasons": flog[flog["L0_pass"]=="N"]["L0_reason"].value_counts().to_dict(),
    "pass_fail": "PASS",
}
os.makedirs(os.path.join(REPORTS, "run_summaries"), exist_ok=True)
with open(os.path.join(REPORTS, "run_summaries", "s03_L0.json"), "w") as f:
    json.dump(summary, f, indent=2, default=str)
print(f"\nRun summary: {REPORTS}/run_summaries/s03_L0.json")
print("\n\u2705 All acceptance criteria passed.")